<a href="https://colab.research.google.com/github/tk-learning-artifacts/ai-accelerator-2026-notebooks/blob/main/student/d1-kernels.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Day 2 · Notebook 2 — Kernels: tiny grids that change pictures

This morning you changed images by hand: cropping, flipping dark and light,
thresholding. This afternoon you'll use a **kernel**: a small grid of numbers
that slides over an image and turns every patch into one new number. Blurring,
sharpening and finding edges are all done this way. So is the first layer of
the networks you'll train on Day 4, except there the network chooses the
numbers.

Same rules as this morning: 🔮 predict before you run, ✏️ replace the `...`,
✅ run the check, 💥 break it when asked. Driver and navigator swap at every
`##` heading.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from skimage import data
from scipy import ndimage


def is_done(*values):
    """True once none of the values is still `...`."""
    return all(v is not ... for v in values)


def check(ok, hint):
    if not ok:
        raise AssertionError("Not yet. " + hint)
    print("✅ Correct!")


def show_row(images, titles, cmap="gray", vmin=None, vmax=None):
    # Without vmin/vmax each picture is stretched to its own darkest and
    # brightest pixel, which hides changes in overall brightness.
    fig, axes = plt.subplots(1, len(images), figsize=(4 * len(images), 4))
    for ax, img, title in zip(np.atleast_1d(axes), images, titles):
        ax.imshow(img, cmap=cmap, vmin=vmin, vmax=vmax)
        ax.set_title(title)
        ax.axis("off")
    plt.show()


photo = data.camera().astype(float)
print("Ready. photo is", photo.shape)

## 1. One window, one number

This is what you did on paper. Put the 3×3 kernel on top of a 3×3 patch,
multiply each pair of numbers that sit on top of each other, and add up the
nine results.

In [ ]:
patch = np.array([
    [10, 10, 10],
    [10, 50, 10],
    [10, 10, 10],
])
blur = np.ones((3, 3)) / 9
print(blur)

🔮 **Predict.** What one number do you get when `blur` sits on `patch`? Work it
out on paper first.

In [ ]:
my_prediction = ""  # @param {type:"string"}

In [ ]:
print("multiplied pairs:\n", patch * blur)
print("sum:", (patch * blur).sum())

The bright 50 in the middle became about 14: the blur kernel replaced the
pixel with the **average** of its neighbourhood, so a bright dot gets pulled
toward its surroundings.

✏️ **Exercise 1a.** Write `apply_at(img, kernel, r, c)`: put the kernel's centre
on pixel `(r, c)` and return the one number. You need the 3×3 patch whose
centre is `(r, c)`: rows `r - 1` to `r + 1`, columns `c - 1` to `c + 1`.
Remember that a slice stops *before* its end.

In [ ]:
def apply_at(img, kernel, r, c):
    return ...

In [ ]:
check(is_done(apply_at(patch, blur, 1, 1)), "apply_at still returns ... .")
check(np.isclose(apply_at(patch, blur, 1, 1), 130 / 9), "On the patch above the answer is 130/9 ≈ 14.4. Is your patch 3×3?")
check(np.isclose(apply_at(photo, blur, 100, 200), photo[99:102, 199:202].mean()),
      "Centre the patch on (r, c): rows r-1 up to and including r+1.")

## 2. Slide the window: convolution

Do that at every pixel and you get a whole new image. Sliding a kernel across
an image like this is called **convolution**.

🔮 **Predict.** The kernel's centre can't sit on the very edge, because part of
the kernel would hang off the image. If we skip the edge pixels, what shape
will the output of a 3×3 kernel on our 512×512 photo be?

In [ ]:
my_prediction = ""  # @param {type:"string"}

✏️ **Exercise 2a.** Fill in the one missing line: store `apply_at` for pixel
`(r, c)` in the right place of `out`. The output is smaller than the input, so
pixel `(1, 1)` of the input goes to `out[0, 0]`.

In [ ]:
def convolve(img, kernel):
    h, w = img.shape
    out = np.zeros((h - 2, w - 2))
    for r in range(1, h - 1):
        for c in range(1, w - 1):
            ...  # one line: put apply_at(...) into out
    return out

In [ ]:
small = photo[200:220, 200:230]
result = convolve(small, blur)
check(result.shape == (18, 28), f"A 20×30 image should give 18×28, but you got {result.shape}.")
check(result.any(), "out is still all zeros. Did you store anything in it?")
check(np.allclose(result, ndimage.correlate(small, blur)[1:-1, 1:-1]),
      "Some values are in the wrong place. Pixel (r, c) goes to out[r - 1, c - 1].")

Now blur the whole photo with your function. It does about 260,000 small
sums, so it takes a second or two.

In [ ]:
blurred = convolve(photo, blur)
show_row([photo, blurred], ["photo", "blurred with your convolve"])

🐛 **Debugging challenge.** Another pair wrote this version. It runs without an
error, but look at the picture it makes: the bottom of the image has a black
stripe. Find the bug and fix it, then run the check.

In [ ]:
def convolve_pair_b(img, kernel):
    h, w = img.shape
    out = np.zeros((h - 2, w - 2))
    for r in range(1, h - 10):
        for c in range(1, w - 1):
            out[r - 1, c - 1] = (img[r - 1:r + 2, c - 1:c + 2] * kernel).sum()
    return out


show_row([convolve_pair_b(photo, blur)], ["pair B's blur"])

In [ ]:
check(convolve_pair_b(photo, blur)[-1].any(), "The bottom row is still black. Which loop decides how far down we go?")

Your loop is clear but slow. Libraries do the same sum in fast compiled code.
From here on we use this version so everyone continues from the same place.

In [ ]:
def fast_convolve(img, kernel):
    """Same answer as convolve, much faster."""
    return ndimage.correlate(img.astype(float), np.asarray(kernel, float))[1:-1, 1:-1]

## 3. Kernel zoo: predict, then reveal

Here are four mystery kernels. Look at the numbers and think about what each
one does to a patch.

In [ ]:
kernel_A = np.array([[0, 0, 0],
                     [0, 1, 0],
                     [0, 0, 0]])
kernel_B = np.array([[-1, 0, 1],
                     [-2, 0, 2],
                     [-1, 0, 1]])
kernel_C = np.array([[0, -1, 0],
                     [-1, 5, -1],
                     [0, -1, 0]])
kernel_D = np.array([[-1, -2, -1],
                     [0, 0, 0],
                     [1, 2, 1]])

🔮 **Predict.** Pick one effect for each kernel from the drop-down boxes,
before you run the reveal. (Hint for B: on a flat patch where all nine pixels
are equal, what does it give? And where the left side is dark and the right
side bright?)

(Run this cell after choosing, so your choices are saved.)

In [ ]:
kernel_A_does = "?"  # @param ["?", "nothing", "blur", "sharpen", "find vertical edges", "find horizontal edges"]
kernel_B_does = "?"  # @param ["?", "nothing", "blur", "sharpen", "find vertical edges", "find horizontal edges"]
kernel_C_does = "?"  # @param ["?", "nothing", "blur", "sharpen", "find vertical edges", "find horizontal edges"]
kernel_D_does = "?"  # @param ["?", "nothing", "blur", "sharpen", "find vertical edges", "find horizontal edges"]

In [ ]:
crop = photo[60:260, 140:340]
outputs = [fast_convolve(crop, k) for k in (kernel_A, kernel_B, kernel_C, kernel_D)]
show_row([crop] + outputs, ["original", "A", "B", "C", "D"])
print("Your predictions:", kernel_A_does, "|", kernel_B_does, "|", kernel_C_does, "|", kernel_D_does)

Compare with your predictions. Middle grey in the B and D pictures means
roughly "nothing changing here"; bright and dark mean a change from dark to
light or light to dark.

✏️ **Exercise 3a.** The numbers in `blur` add up to 1. Make `bright_blur`, a
blur kernel whose numbers add up to **2** (still all equal).

In [ ]:
bright_blur = ...

In [ ]:
check(is_done(bright_blur), "Replace the ... .")
bright_blur = np.asarray(bright_blur, float)
check(bright_blur.shape == (3, 3), "A 3×3 kernel, please.")
check(np.isclose(bright_blur.sum(), 2) and np.allclose(bright_blur, bright_blur[0, 0]),
      "Nine equal numbers that add up to 2.")

🔮 **Predict.** What will `bright_blur` do to the photo's brightness? Then run.

In [ ]:
my_prediction = ""  # @param {type:"string"}

In [ ]:
out = fast_convolve(crop, bright_blur)
print("average brightness before:", round(crop.mean()), " after:", round(out.mean()))
show_row([crop, np.clip(out, 0, 255)], ["original", "bright blur (clipped to 255)"], cmap="gray", vmin=0, vmax=255)

## 4. Challenge: make your own kernel

Here is a test picture with straight lines (a square) and diagonal lines (an
X). Your job: design a 3×3 kernel that lights up the **diagonal** lines and
stays dark on the **straight** ones.

In [ ]:
test = np.zeros((64, 64))
test[8, 8:56] = test[55, 8:56] = 1        # top and bottom of the square
test[8:56, 8] = test[8:56, 55] = 1        # left and right of the square
idx = np.arange(16, 48)
test[idx, idx] = test[idx, 63 - idx] = 1  # the X

on_diagonal = np.zeros_like(test, dtype=bool)
on_diagonal[idx, idx] = on_diagonal[idx, 63 - idx] = True
on_straight = np.zeros_like(test, dtype=bool)
on_straight[8, 8:56] = on_straight[55, 8:56] = on_straight[8:56, 8] = on_straight[8:56, 55] = True


def diagonal_score(kernel):
    """How much stronger the kernel responds on diagonals than on straight lines."""
    response = np.abs(ndimage.correlate(test, np.asarray(kernel, float)))
    return min(response[on_diagonal].mean() / max(response[on_straight].mean(), 1e-9), 100)


show_row([test], ["test picture"])
print("score of the do-nothing kernel A:", round(diagonal_score(kernel_A), 2))

A score of 1 means "no preference". Higher is better, and 100 is the top: a
kernel that gives exactly nothing on straight lines. The edge detectors from
section 3 score about 2.

✏️ **Exercise 4a.** Change the numbers in `my_kernel` until its score is above
**5**. Think about what a diagonal line looks like inside a 3×3 window before
you start guessing.

In [ ]:
my_kernel = np.array([
    [0, 0, 0],
    [0, 1, 0],
    [0, 0, 0],
])
score = diagonal_score(my_kernel)
print("score:", round(score, 2))
show_row([test, np.abs(fast_convolve(test, my_kernel))], ["test", "your kernel's response"], cmap="magma")

In [ ]:
check(np.asarray(my_kernel).shape == (3, 3), "Keep it 3×3.")
check(diagonal_score(my_kernel) > 5, f"Score {diagonal_score(my_kernel):.2f}. Try: big numbers along one diagonal, negative numbers elsewhere.")

🏁 **Leaderboard.** Post your score and your kernel on the class board. Several
pairs may reach 100; ties are broken by the gallery vote below.

Now try your kernel on the tissue from this morning. What does it pick out?

In [ ]:
tissue_grey = data.immunohistochemistry().mean(axis=2)
show_row([tissue_grey, np.abs(fast_convolve(tissue_grey, my_kernel))],
         ["tissue", "your kernel on tissue"], cmap="magma")

🖼️ **Gallery walk.** Leave your best tissue picture on screen. Walk round the
room and vote for the kernel that makes the most useful picture for finding
cells.

## Extension (if you finish early)

Open the Day 2 extension track in Sidecar, or try one of these:

- Make a kernel that responds to the `\` diagonal but not the `/` one. Change
  `diagonal_score` to measure that and test your kernel.
- A 5×5 blur needs a bigger window. Change `apply_at` and `convolve` to work
  for any odd kernel size.
- Time `convolve(photo, blur)` against `fast_convolve(photo, blur)` with
  `%%time`. How many times faster is the library?

In [ ]:
# Your extension work here.